In [66]:
import numpy as np

In [67]:
# suggested values
alpha_val = 0.9
beta_val = 0.2
small_gamma_val = 0.1
lambda_0 = 1
lambda_1 = 5

n = 10
big_T = 100

Plan:
* make transition matrix
* make emission matrix
* make pi?
* simulate (research how)

In [68]:

# C_t = 0: serial stimuli 0
# C_t = 1: serial stimuli 1
# C_t = 2: parallel both stimuli 0 & 1

def create_transition_matrix(gamma: float, beta: float):
    Gamma = np.array([[1 - gamma,  0,          gamma], 
                      [0,          1 - gamma,  gamma], 
                      [beta/2,     beta/2,     1 - beta]])
    return Gamma

## Emission - conditional $Z_{t,i}$
Since our model has multiple layers, we don't directly have an emission matrix from $C_t$ to $X_{t,i}$. Instead we need to go through $Z_{t,i}$ up to $C_t$, which is given as the conditional:
$$
P(Z_{t,i} = 1 | C_t = c) = \begin{cases}
1 - \alpha & \text{if } c = 0 \\
\alpha & \text{if } c = 1 \\
0.5 & \text{if } c = 2
\end{cases}
$$


In [69]:
def p_z_given_c(alpha, c: int): # z_{t,i} = 1 | C_t = c
    if c == 0:
        return 1 - alpha
    elif c == 1:
        return alpha
    elif c == 2:
        return 0.5
    else:
        raise ValueError("Error with input, only c = 0, c = 1 or c = 2 accepted.")

In [70]:
# p_z_given_c(alpha=alpha_val, c=2)

In [71]:
test_matrix = create_transition_matrix(gamma=small_gamma_val, beta=beta_val)
print(test_matrix)

[[0.9 0.  0.1]
 [0.  0.9 0.1]
 [0.1 0.1 0.8]]


## Random Walk for $n = 10$, and $T = 100$

In [72]:

# * suggested values
# alpha_val = 0.9
# beta_val = 0.2
# small_gamma_val = 0.1
# lambda_0 = 1
# lambda_1 = 5

# n = 10
# big_T = 100

def c(gamma, beta, T):
    Gamma = create_transition_matrix(gamma=gamma, beta=beta)
    C = np.empty(T, dtype=int) # empty matrix to hold simulated values
    C[0] = 2 # first entry is always 2
    
    c_states = np.array([0,1,2], dtype=int) # possible states C_t can take on
    
    for t in range(1, T):
        # depending on the value the previous C had, we do a choice for where to go next given the distribution
        # so if we start at 2, our C[T-1] = 2, and so we'd access p=Gamma[2], and sample a random choice of those (giving us [0,1,2] as possible outcomes)
        # and select that value as the next C[T].
        C[t] = np.random.choice(a=c_states, p=Gamma[C[t-1]]) 
        
    return C
        

In [73]:
C = c(gamma=small_gamma_val, beta=beta_val, T=big_T)
C

array([2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 2, 2, 2, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2,
       2, 2, 0, 0, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2,
       2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2])

In [ ]:
def z_given_c(C, alpha, n):
    T = len(C) # to ensure we don't give a new T which would break our previous (shapes)
    Z = np.empty(shape=(T, n), dtype=int) # before we only had one C per time slice, but now we have a Z per neuron, of which there are n, so we need a matrix instead of an array
    for t in range(0, T):
        z_prob = p_z_given_c(alpha=alpha, c=C[t]) # all Z's have the same C (share parent), so we only get a single C per row in matri x
        Z[t, :] = np.random.binomial(n=1, p=z_prob, size=n) # and now we do a random draw per z, that is for each row we do n random bernoulli samples (z \in \brc{0, 1})
    return Z